# CrossDocked → EMDB Pipeline

CrossDocked is entirely **X-ray crystallography** — no PDB entries have direct EMDB links.  
Instead we go via **UniProt**: find cryo-EM structures of the *same protein*, then get their EMDB density maps.

```
CrossDocked PDB ID
    └─ RCSB GraphQL (batch)  →  UniProt accession(s)
           └─ RCSB search API  →  EM PDB IDs (same UniProt + method=EM)
                  └─ RCSB entry API  →  EMDB accession(s)
                         └─ EBI FTP  →  .map.gz download
```

Results are checkpointed to `notebook/data/` at each stage so the notebook can be interrupted and resumed.

In [9]:
import os, json, time, sys
import requests
import torch
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

REPO_ROOT = Path(os.getcwd()).parent          # .../Voxbind
DATA_DIR  = REPO_ROOT / "voxbind" / "dataset" / "data"
OUT_DIR   = Path(os.getcwd()) / "data"
EMDB_DIR  = OUT_DIR / "emdb"
EMDB_DIR.mkdir(parents=True, exist_ok=True)

print(f"Data dir : {DATA_DIR}")
print(f"Out dir  : {OUT_DIR}")

Data dir : /home/shpark/prj-ligand/Voxbind/voxbind/dataset/data
Out dir  : /home/shpark/prj-ligand/Voxbind/notebook/data


## Step 1 — Extract unique PDB IDs from CrossDocked

Each sample's `pocket["id"]` encodes the PDB ID:
```
GLF_MYCTU_1_399_0 / 4rpg_A_rec_..._pocket10.pdb
                     ^^^^  ← PDB ID
```

In [10]:
def extract_pdb_id(pocket_id: str) -> str:
    return pocket_id.split("/")[1].split("_")[0].upper()


# pdb_to_pocket_ids: PDB ID → list of (pocket_id, split) for traceability
pdb_to_pocket_ids: dict[str, list] = {}

for split_name, pt_file in [("train", "data_train.pt"), ("test", "data_test.pt")]:
    pt_path = DATA_DIR / pt_file
    if not pt_path.exists():
        print(f"[SKIP] {pt_file} not found")
        continue
    samples = torch.load(pt_path, weights_only=False)
    for pocket_dict, _ in samples:
        pid = extract_pdb_id(pocket_dict["id"])
        pdb_to_pocket_ids.setdefault(pid, []).append((pocket_dict["id"], split_name))
    print(f"Loaded {pt_file}: {len(samples):,} samples")

pdb_ids = sorted(pdb_to_pocket_ids)
print(f"\nUnique PDB IDs: {len(pdb_ids):,}")

Loaded data_train.pt: 99,981 samples
Loaded data_test.pt: 100 samples

Unique PDB IDs: 14,735


## Step 2 — Batch-resolve PDB IDs → UniProt via RCSB GraphQL

RCSB GraphQL accepts up to ~200 PDB IDs per request.  
Results are cached to `pdb_to_uniprot.json`.

In [11]:
UNIPROT_FILE  = OUT_DIR / "pdb_to_uniprot.json"
GRAPHQL_URL   = "https://data.rcsb.org/graphql"
BATCH_SIZE    = 150   # IDs per GraphQL request

# Load cache
if UNIPROT_FILE.exists():
    with open(UNIPROT_FILE) as f:
        pdb_to_uniprot: dict[str, list[str]] = json.load(f)
    print(f"Loaded cached UniProt mapping ({len(pdb_to_uniprot):,} entries)")
else:
    pdb_to_uniprot = {}

to_resolve = [p for p in pdb_ids if p not in pdb_to_uniprot]
print(f"Need to resolve: {len(to_resolve):,} PDB IDs")


def fetch_uniprots_batch(batch: list[str], session: requests.Session) -> dict[str, list[str]]:
    """GraphQL query for UniProt IDs for a batch of PDB IDs."""
    ids_str = json.dumps(batch)
    query = f"""
    {{
      entries(entry_ids: {ids_str}) {{
        rcsb_id
        polymer_entities {{
          rcsb_polymer_entity_container_identifiers {{
            reference_sequence_identifiers {{
              database_name
              database_accession
            }}
          }}
        }}
      }}
    }}
    """
    r = session.post(GRAPHQL_URL, json={"query": query}, timeout=30)
    r.raise_for_status()
    result = {}
    for entry in r.json().get("data", {}).get("entries") or []:
        pid  = entry["rcsb_id"]
        unis = []
        for entity in entry.get("polymer_entities") or []:
            for ref in (entity
                        .get("rcsb_polymer_entity_container_identifiers", {})
                        .get("reference_sequence_identifiers") or []):
                if ref.get("database_name") == "UniProt":
                    unis.append(ref["database_accession"])
        result[pid] = list(dict.fromkeys(unis))  # deduplicated, order-preserved
    # Mark PDB IDs with no entry (e.g., obsolete) as resolved with empty list
    for pid in batch:
        if pid not in result:
            result[pid] = []
    return result


if to_resolve:
    session = requests.Session()
    batches  = [to_resolve[i:i+BATCH_SIZE] for i in range(0, len(to_resolve), BATCH_SIZE)]
    n_done   = 0

    for batch in batches:
        try:
            result = fetch_uniprots_batch(batch, session)
            pdb_to_uniprot.update(result)
        except Exception as e:
            print(f"  [WARN] batch failed: {e}")
        n_done += len(batch)
        if n_done % (BATCH_SIZE * 5) == 0:
            with open(UNIPROT_FILE, "w") as f:
                json.dump(pdb_to_uniprot, f)
            print(f"  [{n_done}/{len(to_resolve)}] checkpoint")

    with open(UNIPROT_FILE, "w") as f:
        json.dump(pdb_to_uniprot, f, indent=2)
    print(f"Done. Saved to {UNIPROT_FILE}")
else:
    print("All cached.")

Need to resolve: 14,735 PDB IDs
  [750/14735] checkpoint
  [1500/14735] checkpoint
  [2250/14735] checkpoint
  [3000/14735] checkpoint
  [3750/14735] checkpoint
  [4500/14735] checkpoint
  [5250/14735] checkpoint
  [6000/14735] checkpoint
  [6750/14735] checkpoint
  [7500/14735] checkpoint
  [8250/14735] checkpoint
  [9000/14735] checkpoint
  [9750/14735] checkpoint
  [10500/14735] checkpoint
  [11250/14735] checkpoint
  [12000/14735] checkpoint
  [12750/14735] checkpoint
  [13500/14735] checkpoint
  [14250/14735] checkpoint
Done. Saved to /home/shpark/prj-ligand/Voxbind/notebook/data/pdb_to_uniprot.json


In [12]:
# Summary
uniprot_to_pdbs: dict[str, list[str]] = {}
for pid, unis in pdb_to_uniprot.items():
    for u in unis:
        uniprot_to_pdbs.setdefault(u, []).append(pid)

no_uniprot = [p for p, v in pdb_to_uniprot.items() if not v]
print(f"PDB IDs with UniProt    : {len(pdb_to_uniprot) - len(no_uniprot):,}")
print(f"PDB IDs without UniProt : {len(no_uniprot):,}  (synthetic / obsolete)")
print(f"Unique UniProt IDs      : {len(uniprot_to_pdbs):,}")

PDB IDs with UniProt    : 14,685
PDB IDs without UniProt : 50  (synthetic / obsolete)
Unique UniProt IDs      : 2,567


## Step 3 — Find cryo-EM PDB structures for each UniProt

Searches RCSB for entries with `method = ELECTRON MICROSCOPY` sharing the same UniProt.  
Results cached to `uniprot_to_em_pdbs.json`.

In [13]:
EM_PDBS_FILE = OUT_DIR / "uniprot_to_em_pdbs.json"
RCSB_SEARCH  = "https://search.rcsb.org/rcsbsearch/v2/query"

if EM_PDBS_FILE.exists():
    with open(EM_PDBS_FILE) as f:
        uniprot_to_em_pdbs: dict[str, list[str]] = json.load(f)
    print(f"Loaded cached EM PDB mapping ({len(uniprot_to_em_pdbs):,} entries)")
else:
    uniprot_to_em_pdbs = {}

all_uniprots   = sorted(uniprot_to_pdbs)
to_search      = [u for u in all_uniprots if u not in uniprot_to_em_pdbs]
print(f"Need to search: {len(to_search):,} UniProt IDs")


def search_em_pdbs(uniprot: str, session: requests.Session) -> tuple[str, list[str]]:
    """Find PDB IDs solved by EM for a given UniProt accession."""
    query = {
        "query": {
            "type": "group",
            "logical_operator": "and",
            "nodes": [
                {"type": "terminal", "service": "text", "parameters": {
                    "attribute": "rcsb_polymer_entity_container_identifiers"
                                 ".reference_sequence_identifiers.database_accession",
                    "operator": "exact_match",
                    "value": uniprot,
                }},
                {"type": "terminal", "service": "text", "parameters": {
                    "attribute": "exptl.method",
                    "operator": "exact_match",
                    "value": "ELECTRON MICROSCOPY",
                }},
            ],
        },
        "return_type": "entry",
        "request_options": {"results_verbosity": "compact"},
    }
    try:
        r = session.post(RCSB_SEARCH, json=query, timeout=15)
        if r.status_code == 204:   # no results
            return uniprot, []
        r.raise_for_status()
        hits = r.json().get("result_set", [])
        return uniprot, hits  # compact mode returns a list of PDB ID strings
    except Exception:
        return uniprot, []


MAX_WORKERS = 16
SAVE_EVERY  = 200

if to_search:
    session = requests.Session()
    n_done  = 0

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = {pool.submit(search_em_pdbs, u, session): u for u in to_search}
        for fut in as_completed(futures):
            uni, em_pdbs = fut.result()
            uniprot_to_em_pdbs[uni] = em_pdbs
            n_done += 1
            if n_done % SAVE_EVERY == 0:
                with open(EM_PDBS_FILE, "w") as f:
                    json.dump(uniprot_to_em_pdbs, f)
                print(f"  [{n_done}/{len(to_search)}] checkpoint")

    with open(EM_PDBS_FILE, "w") as f:
        json.dump(uniprot_to_em_pdbs, f, indent=2)
    print(f"Done. Saved to {EM_PDBS_FILE}")
else:
    print("All cached.")

uniprots_with_em = {u: v for u, v in uniprot_to_em_pdbs.items() if v}
print(f"\nUniProts with EM structures : {len(uniprots_with_em):,}")

Need to search: 2,567 UniProt IDs
  [200/2567] checkpoint
  [400/2567] checkpoint
  [600/2567] checkpoint
  [800/2567] checkpoint
  [1000/2567] checkpoint
  [1200/2567] checkpoint
  [1400/2567] checkpoint
  [1600/2567] checkpoint
  [1800/2567] checkpoint
  [2000/2567] checkpoint
  [2200/2567] checkpoint
  [2400/2567] checkpoint
Done. Saved to /home/shpark/prj-ligand/Voxbind/notebook/data/uniprot_to_em_pdbs.json

UniProts with EM structures : 442


## Step 4 — Resolve EM PDB IDs → EMDB accessions

Uses `pdbx_database_related` from the RCSB entry API.  
Results cached to `pdb_to_emdb.json`.

In [14]:
EMDB_FILE = OUT_DIR / "pdb_to_emdb.json"

if EMDB_FILE.exists():
    with open(EMDB_FILE) as f:
        em_pdb_to_emdb: dict[str, list[str]] = json.load(f)
    print(f"Loaded cached EMDB mapping ({len(em_pdb_to_emdb):,} entries)")
else:
    em_pdb_to_emdb = {}

# Collect all unique EM PDB IDs
all_em_pdb_ids = sorted({pid for ids in uniprots_with_em.values() for pid in ids})
to_resolve     = [p for p in all_em_pdb_ids if p not in em_pdb_to_emdb]
print(f"Unique EM PDB IDs  : {len(all_em_pdb_ids):,}")
print(f"Need to resolve    : {len(to_resolve):,}")


def fetch_emdb_ids(pdb_id: str, session: requests.Session) -> tuple[str, list[str]]:
    try:
        r = session.get(
            f"https://data.rcsb.org/rest/v1/core/entry/{pdb_id}", timeout=15
        )
        if r.status_code != 200:
            return pdb_id, []
        return pdb_id, [
            rel["db_id"]
            for rel in r.json().get("pdbx_database_related", [])
            if rel.get("db_name") == "EMDB"
        ]
    except Exception:
        return pdb_id, []


if to_resolve:
    session = requests.Session()
    n_done  = 0

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = {pool.submit(fetch_emdb_ids, p, session): p for p in to_resolve}
        for fut in as_completed(futures):
            pid, emdb_ids = fut.result()
            em_pdb_to_emdb[pid] = emdb_ids
            n_done += 1
            if n_done % SAVE_EVERY == 0:
                with open(EMDB_FILE, "w") as f:
                    json.dump(em_pdb_to_emdb, f)
                print(f"  [{n_done}/{len(to_resolve)}] checkpoint")

    with open(EMDB_FILE, "w") as f:
        json.dump(em_pdb_to_emdb, f, indent=2)
    print(f"Done. Saved to {EMDB_FILE}")
else:
    print("All cached.")

Loaded cached EMDB mapping (14,735 entries)
Unique EM PDB IDs  : 1,954
Need to resolve    : 1,954
  [200/1954] checkpoint
  [400/1954] checkpoint
  [600/1954] checkpoint
  [800/1954] checkpoint
  [1000/1954] checkpoint
  [1200/1954] checkpoint
  [1400/1954] checkpoint
  [1600/1954] checkpoint
  [1800/1954] checkpoint
Done. Saved to /home/shpark/prj-ligand/Voxbind/notebook/data/pdb_to_emdb.json


## Step 5 — Build final CrossDocked PDB → EMDB mapping

In [15]:
FINAL_FILE = OUT_DIR / "crossdocked_pdb_to_emdb.json"

crossdocked_to_emdb: dict[str, list[str]] = {}

for cd_pid in pdb_ids:
    emdb_ids = set()
    for uni in pdb_to_uniprot.get(cd_pid, []):
        for em_pid in uniprot_to_em_pdbs.get(uni, []):
            emdb_ids.update(em_pdb_to_emdb.get(em_pid, []))
    crossdocked_to_emdb[cd_pid] = sorted(emdb_ids)

with open(FINAL_FILE, "w") as f:
    json.dump(crossdocked_to_emdb, f, indent=2)

# Summary
with_emdb    = {k: v for k, v in crossdocked_to_emdb.items() if v}
without_emdb = {k for k, v in crossdocked_to_emdb.items() if not v}

print(f"Total CrossDocked PDB IDs : {len(crossdocked_to_emdb):,}")
print(f"  With EMDB map           : {len(with_emdb):,}")
print(f"  Without EMDB map        : {len(without_emdb):,}")
print()
print("Examples with EMDB:")
for pid, emdb_ids in list(with_emdb.items())[:10]:
    n_pockets = len(pdb_to_pocket_ids.get(pid, []))
    print(f"  {pid}  →  {emdb_ids}  ({n_pockets} pocket(s) in dataset)")

Total CrossDocked PDB IDs : 14,735
  With EMDB map           : 3,363
  Without EMDB map        : 11,372

Examples with EMDB:
  132L  →  ['EMD-18669', 'EMD-1981', 'EMD-1982', 'EMD-50432', 'EMD-50433', 'EMD-61056', 'EMD-61057', 'EMD-62748']  (4 pocket(s) in dataset)
  1AJ6  →  ['EMD-14570', 'EMD-14572', 'EMD-14573', 'EMD-14574', 'EMD-18342', 'EMD-18565', 'EMD-18566', 'EMD-18567', 'EMD-18592', 'EMD-4909', 'EMD-4910', 'EMD-4912', 'EMD-4913']  (1 pocket(s) in dataset)
  1AJV  →  ['EMD-13139', 'EMD-13156', 'EMD-21582', 'EMD-21584', 'EMD-22899', 'EMD-22900', 'EMD-22901', 'EMD-25074', 'EMD-25165', 'EMD-7031', 'EMD-7032']  (8 pocket(s) in dataset)
  1AJX  →  ['EMD-13139', 'EMD-13156', 'EMD-21582', 'EMD-21584', 'EMD-22899', 'EMD-22900', 'EMD-22901', 'EMD-25074', 'EMD-25165', 'EMD-7031', 'EMD-7032']  (8 pocket(s) in dataset)
  1AS0  →  ['EMD-20812', 'EMD-21243', 'EMD-24789', 'EMD-24790', 'EMD-25819', 'EMD-25820', 'EMD-25821', 'EMD-25822', 'EMD-25823', 'EMD-45425']  (16 pocket(s) in dataset)
  1AS

## Step 6 — Download EMDB maps

Downloads `.map.gz` files from the EBI FTP server. Files are cached — already downloaded maps are skipped.

In [16]:
def emdb_num(emdb_id: str) -> str:
    """'EMD-5778' → '5778'"""
    return emdb_id.split("-")[-1]


def download_emdb_map(emdb_id: str, out_dir: Path, session: requests.Session) -> Path | None:
    num    = emdb_num(emdb_id)
    out_gz = out_dir / f"emd_{num}.map.gz"
    if out_gz.exists():
        return out_gz
    url = (
        f"https://ftp.ebi.ac.uk/pub/databases/emdb/structures/"
        f"EMD-{num}/map/emd_{num}.map.gz"
    )
    try:
        r = session.get(url, stream=True, timeout=600)
        r.raise_for_status()
        with open(out_gz, "wb") as f:
            for chunk in r.iter_content(chunk_size=4 * 1024 * 1024):
                f.write(chunk)
        return out_gz
    except Exception as e:
        print(f"  [ERROR] {emdb_id}: {e}")
        if out_gz.exists():
            out_gz.unlink()
        return None


all_emdb_ids   = sorted({eid for ids in with_emdb.values() for eid in ids})
already_cached = [e for e in all_emdb_ids
                  if (EMDB_DIR / f"emd_{emdb_num(e)}.map.gz").exists()]
to_download    = [e for e in all_emdb_ids if e not in already_cached]

print(f"Unique EMDB maps to download : {len(all_emdb_ids):,}")
print(f"  Already cached             : {len(already_cached):,}")
print(f"  Need download              : {len(to_download):,}")

Unique EMDB maps to download : 2,394
  Already cached             : 0
  Need download              : 2,394


In [17]:
session = requests.Session()
failed  = []

for i, emdb_id in enumerate(to_download, 1):
    num = emdb_num(emdb_id)
    url = (f"https://ftp.ebi.ac.uk/pub/databases/emdb/structures/"
           f"EMD-{num}/map/emd_{num}.map.gz")
    try:
        head  = session.head(url, timeout=10)
        size  = int(head.headers.get("content-length", 0)) // 1_000_000
        sstr  = f"{size} MB"
    except Exception:
        sstr  = "? MB"

    print(f"[{i}/{len(to_download)}] {emdb_id}  ({sstr}) ...", end=" ", flush=True)
    path = download_emdb_map(emdb_id, EMDB_DIR, session)
    if path:
        actual = path.stat().st_size // 1_000_000
        print(f"OK ({actual} MB)")
    else:
        print("FAILED")
        failed.append(emdb_id)

print(f"\nFailed: {failed if failed else 'none'}")

[1/2394] EMD-0013  (69 MB) ... OK (69 MB)
[2/2394] EMD-0014  (5 MB) ... OK (5 MB)
[3/2394] EMD-0031  (16 MB) ... OK (16 MB)
[4/2394] EMD-0038  (52 MB) ... OK (52 MB)
[5/2394] EMD-0051  (62 MB) ... OK (62 MB)
[6/2394] EMD-0095  (2 MB) ... OK (2 MB)
[7/2394] EMD-0096  (6 MB) ... OK (6 MB)
[8/2394] EMD-0097  (3 MB) ... OK (3 MB)
[9/2394] EMD-0132  (12 MB) ... OK (12 MB)
[10/2394] EMD-0138  (61 MB) ... OK (61 MB)
[11/2394] EMD-0148  (57 MB) ... OK (57 MB)
[12/2394] EMD-0201  (277 MB) ... OK (277 MB)
[13/2394] EMD-0202  (170 MB) ... OK (170 MB)
[14/2394] EMD-0247  (2 MB) ... OK (2 MB)
[15/2394] EMD-0339  (51 MB) ... OK (51 MB)
[16/2394] EMD-0345  (62 MB) ... OK (62 MB)
[17/2394] EMD-0346  (62 MB) ... OK (62 MB)
[18/2394] EMD-0347  (2 MB) ... OK (2 MB)
[19/2394] EMD-0357  (5 MB) ... OK (5 MB)
[20/2394] EMD-0359  (5 MB) ... OK (5 MB)
[21/2394] EMD-0362  (5 MB) ... OK (5 MB)
[22/2394] EMD-0363  (4 MB) ... OK (4 MB)
[23/2394] EMD-0364  (4 MB) ... OK (4 MB)
[24/2394] EMD-0365  (3 MB) ... OK (3 M

KeyboardInterrupt: 

## Lookup helper

Given any CrossDocked `pocket["id"]` string (or a PDB ID directly), find the available EMDB maps.

In [ ]:
def get_emdb_for_pocket(pocket_id: str) -> dict:
    """
    Given a CrossDocked pocket_id, return:
      {
        'pdb_id'    : str,           # CrossDocked PDB ID
        'emdb_ids'  : list[str],     # e.g. ['EMD-35526']
        'local_maps': list[Path],    # downloaded .map.gz paths
      }
    """
    pdb_id   = extract_pdb_id(pocket_id)
    emdb_ids = crossdocked_to_emdb.get(pdb_id, [])
    local    = [
        EMDB_DIR / f"emd_{emdb_num(e)}.map.gz"
        for e in emdb_ids
        if (EMDB_DIR / f"emd_{emdb_num(e)}.map.gz").exists()
    ]
    return dict(pdb_id=pdb_id, emdb_ids=emdb_ids, local_maps=local)


# ── Demo: first CrossDocked pocket with an EMDB hit ──────────────────
example_pdb = next(iter(with_emdb))
example_pid = pdb_to_pocket_ids[example_pdb][0][0]

result = get_emdb_for_pocket(example_pid)
print(f"pocket_id  : {example_pid}")
print(f"PDB ID     : {result['pdb_id']}")
print(f"EMDB IDs   : {result['emdb_ids']}")
print(f"Local maps : {result['local_maps']}")